# 00 Dataset Audit and Cleaning

This notebook audits datasets A, B, and C, applies conservative cleaning rules, and exports cleaned CSV files for downstream training.

In [8]:
from pathlib import Path
import json
import pandas as pd
import sys

SCRIPTS_DIR = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/scripts')
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from preprocessing import CleaningConfig, clean_dataset, build_audit_summary, export_clean_dataset, export_audit_summary, export_split_summary

## Cleaning Policy

- remove duplicate business keys
- parse and standardize month values
- use `demand_units_clean` for dataset C by default
- clip negative demand/exogenous fields to zero
- do not cap A/B outliers unless explicitly enabled

In [9]:
config = CleaningConfig(cap_outliers_for_ab=False, outlier_iqr_k=3.0)
config

CleaningConfig(use_scenario_clean_target=True, clip_negative_demand=True, cap_outliers_for_ab=False, outlier_iqr_k=3.0)

In [10]:
audit_rows = []
cleaned = {}

for dataset in ['A', 'B', 'C']:
    df = clean_dataset(dataset, config)
    cleaned[dataset] = df
    summary = build_audit_summary(df, dataset)
    export_clean_dataset(df, dataset)
    export_audit_summary(summary, dataset)
    export_split_summary(df, dataset)
    audit_rows.append({
        'dataset': dataset,
        'rows': summary['rows'],
        'n_series': summary['n_series'],
        'n_months': summary['n_months'],
        'duplicate_key_rows': summary['duplicate_key_rows'],
        'outlier_rate': summary.get('c_is_outlier_rate', 0.0),
    })

pd.DataFrame(audit_rows)

,dataset,rows,n_series,n_months,duplicate_key_rows,outlier_rate
0,A,3708,103,36,0,0.000000
1,B,3708,103,36,0,0.000000
2,C,222480,6180,36,0,0.011929


## Inspect Cleaned Samples

In [11]:
cleaned['A'].head()

,dataset,month,fg_code,fg_name,fg_category,series_id,demand_units,demand_units_raw,demand_units_clean_source,scenario_id,scenario_split,c_is_outlier
0,A,2023-02-01,FG001,Baby Soap 50g,Soap,FG001,27456,27456,27456,-1,all,0
1,A,2023-03-01,FG001,Baby Soap 50g,Soap,FG001,27316,27316,27316,-1,all,0
2,A,2023-04-01,FG001,Baby Soap 50g,Soap,FG001,25194,25194,25194,-1,all,0
3,A,2023-05-01,FG001,Baby Soap 50g,Soap,FG001,22610,22610,22610,-1,all,0
4,A,2023-06-01,FG001,Baby Soap 50g,Soap,FG001,22763,22763,22763,-1,all,0


In [12]:
cleaned['B'].head()

,dataset,month,fg_code,fg_name,fg_category,series_id,demand_units,demand_units_raw,demand_units_clean_source,scenario_id,...,on_hand_inventory,stockout_days,promotion_flag,price_or_discount,lead_time_days,supplier_otif,inbound_po_qty,open_sales_orders,returns_qty,holiday_flag
0,B,2023-02-01,FG001,Baby Soap 50g,Soap,FG001,27456,27456,27456,-1,...,46629,1,0,0.0072,11,0.9619,13409,2739,225,0
1,B,2023-03-01,FG001,Baby Soap 50g,Soap,FG001,27316,27316,27316,-1,...,49610,0,0,0.0157,14,0.9445,14050,7145,223,0
2,B,2023-04-01,FG001,Baby Soap 50g,Soap,FG001,25194,25194,25194,-1,...,54035,0,1,0.2155,11,0.9748,13851,5163,238,1
3,B,2023-05-01,FG001,Baby Soap 50g,Soap,FG001,22610,22610,22610,-1,...,51854,0,0,0.0168,8,0.9807,6593,6067,183,0
4,B,2023-06-01,FG001,Baby Soap 50g,Soap,FG001,22763,22763,22763,-1,...,41490,0,0,0.0070,16,0.9363,11456,2238,171,0


In [13]:
cleaned['C'][['month','fg_code','scenario_id','scenario_split','demand_units_raw','demand_units_clean_source','demand_units','c_is_outlier']].head()

,month,fg_code,scenario_id,scenario_split,demand_units_raw,demand_units_clean_source,demand_units,c_is_outlier
0,2023-02-01,FG001,0,train,30092,30092,30092,0
1,2023-03-01,FG001,0,train,23227,23227,23227,0
2,2023-04-01,FG001,0,train,31023,31023,31023,0
3,2023-05-01,FG001,0,train,26841,26841,26841,0
4,2023-06-01,FG001,0,train,26301,26301,26301,0


## Saved Outputs

In [14]:
from preprocessing import CLEAN_DIR, AUDIT_DIR
sorted(str(p) for p in CLEAN_DIR.glob('*')) + sorted(str(p) for p in AUDIT_DIR.glob('*'))

['/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/cleaned/dataset_a_clean.csv',
 '/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/cleaned/dataset_b_clean.csv',
 '/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/cleaned/dataset_c_clean.csv',
 '/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/cleaned/dataset_p_clean.csv',
 '/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/cleaned/dataset_w_clean.csv',
 '/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/audit/dataset_a_audit_summary.json',
 '/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/audit/dataset_a_split_summary.csv',
 '/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/audit/dataset_b_audit_summary.json',
 '/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/audit/dataset_b_split_summary.csv',
 '/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/ou